# TextZoom baseline check

1. Reproduce the **BICUBIC** rows of Table 4 (arXiv:2005.03341) with the migrated code. This validates data, recognizers and the migration.
2. Evaluate a **community-trained TSRN** checkpoint as a possible pruning starting point.
   2b. Train a **clean TSRN baseline** with the original recipe on the 95% training split (resumable; A100 recommended).
3. Time a few minutes of **TSRN** training to estimate the cost of the full 500 epochs.
4. Profile **TSRN** inference: time share of GRUs vs convolutions on GPU and CPU (needs section 2 for the checkpoint path).
5. **Pruning pilot**: one 20% pruning step on TSRN's convolutions, short fine-tune, then accuracy, PSNR, size and speed (needs section 2).
6. **RQ1 trajectory**: prune step by step to 87.5% of the channels, recording PSNR and OCR after every step (resumable).
7. **RQ1/RQ2 figures**: PSNR change vs OCR change, cost vs speed, and both stop rules applied to the recorded curve.
8. **Same-session latency** of every trajectory model, for the paper.
9. **Re-score** every model with the three readers, image by image (steadier OCR stop rule, McNemar tests).
10. **Example figure**: images where pruning changes what CRNN reads.

Runtime → Change runtime type → **T4 GPU** before running (**A100** for section 2b if available; latency in section 8 always on a T4).

**Results on Drive** (`MyDrive/textzoom/`):
```
data/  aster.pth.tar  moran.pth  crnn.pth  tsrn_community.pth   inputs, downloaded once
experiments/
  baseline/<run name>/         section 2b (checkpoints/final.pth = the clean baseline)
  pilot/<run name>/            section 5
  rq1_trajectory/<run name>/   sections 6-8 (each run folder has a README.md listing its files)
```
Run names describe the settings, e.g. `tsrn-clean_conv-only_taylor_linear-7steps-to88pct_round8_ft20ep-es3_kd0.1_select-psnr_seed0` (runs from the community checkpoint start with `tsrn_conv-only_`).

In [ ]:
# Closest stack to the SwinIR pruning project that installs on Python 3.13.
!nvidia-smi && python --version
!pip install -q torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0

In [ ]:
REPO = '/content/TextZoomPruning'
%cd /content
!git clone -b migrate-torch2 https://github.com/Maouhoub/TextZoomPruning.git
%cd {REPO}
!pip install -q -r requirements.txt

import torch, torchvision, numpy
print(torch.__version__, torchvision.__version__, numpy.__version__, torch.cuda.get_device_name(0))

In [ ]:
# Downloads are kept on Drive so later sessions skip them.
from google.colab import drive
drive.mount('/content/drive')

import os
STORE = '/content/drive/MyDrive/textzoom'
os.makedirs(STORE, exist_ok=True)

In [ ]:
%cd {STORE}
# TextZoom LMDB (official Google Drive folder from the TextZoom README)
![ -f .data_done ] || (gdown --folder https://drive.google.com/drive/folders/1WRVy-fC_KrembPkaI68uqQ9wyaptibMh -O data && touch .data_done)
# Pretrained recognizers, used as-is
![ -f aster.pth.tar ] || wget -q -O aster.pth.tar https://github.com/ayumiymk/aster.pytorch/releases/download/v1.0/demo.pth.tar
![ -f moran.pth ] || gdown 1IDvT51MXKSseDq3X57uPjOzeSYI09zip -O moran.pth
![ -f crnn.pth ] || wget -q -O crnn.pth "https://www.dropbox.com/s/dboqjk20qjkpta3/crnn.pth?dl=1"
!ls -la {STORE}

# LMDB reads from the Drive mount are slow, so work on a local copy.
!rm -rf /content/data && cp -r {STORE}/data /content/data
!find /content/data -name data.mdb

In [ ]:
# Point the config at the data and recognizer weights.
import glob, yaml

lmdb_dirs = sorted(os.path.dirname(p) for p in glob.glob('/content/data/**/data.mdb', recursive=True))
by_name = {os.path.basename(d): d for d in lmdb_dirs}
print(by_name)
TEST = {split: by_name[split] for split in ('easy', 'medium', 'hard')}

cfg_path = f'{REPO}/src/config/super_resolution.yaml'
cfg = yaml.safe_load(open(cfg_path))
cfg['TRAIN']['train_data_dir'] = [by_name['train1'], by_name['train2']]
cfg['TRAIN']['VAL']['val_data_dir'] = [TEST['hard'], TEST['easy'], TEST['medium']]
cfg['TRAIN']['VAL']['rec_pretrained'] = f'{STORE}/aster.pth.tar'
cfg['TRAIN']['VAL']['moran_pretrained'] = f'{STORE}/moran.pth'
cfg['TRAIN']['VAL']['crnn_pretrained'] = f'{STORE}/crnn.pth'
yaml.safe_dump(cfg, open(cfg_path, 'w'), sort_keys=False)
print(open(cfg_path).read())

## 1. BICUBIC sanity check (no training)

Our accuracy should be within ~1 point of the paper. If ASTER/MORAN are off by more, the `align_corners` choice is the first suspect.

In [ ]:
import ast, subprocess

SPLITS = ('easy', 'medium', 'hard')
SPLIT_SIZES = (1619, 1411, 1343)  # Table 1, used for the paper's weighted average
PAPER_BICUBIC = {'aster': (64.7, 42.4, 31.2), 'moran': (60.6, 37.9, 30.8), 'crnn': (36.4, 21.1, 21.1)}  # Table 4
PAPER_TSRN = {'aster': (75.1, 56.3, 40.1), 'moran': (70.1, 53.3, 37.9), 'crnn': (52.5, 38.2, 31.4)}  # Table 4
PAPER_BICUBIC_PSNR = (22.35, 18.98, 19.39)  # Table 11


def weighted_average(per_split):
    return sum(a * n for a, n in zip(per_split, SPLIT_SIZES)) / sum(SPLIT_SIZES)


def evaluate(model_args, paper_acc):
    """Run main.py --test for every recognizer and split; print accuracy next to the paper."""
    results = {}
    for rec, paper_row in paper_acc.items():
        ours_row = []
        for split, paper in zip(SPLITS, paper_row):
            run = subprocess.run(['python', 'main.py', '--test', '--rec', rec, '--batch_size', '256',
                                  '--test_data_dir', TEST[split], *model_args],
                                 cwd=f'{REPO}/src', capture_output=True, text=True)
            lines = run.stdout.strip().splitlines()
            if run.returncode != 0 or not lines or not lines[-1].startswith('{'):
                print(run.stdout[-2000:], run.stderr[-4000:])
                raise RuntimeError(f'{rec}/{split} failed')
            result = ast.literal_eval(lines[-1])
            results[(rec, split)] = result
            acc = result['accuracy'][split] * 100
            ours_row.append(acc)
            print(f'{rec:6s} {split:7s} acc {acc:5.1f} (paper {paper:5.1f})  psnr {result["psnr_avg"]:.2f}')
        print(f'{rec:6s} average acc {weighted_average(ours_row):5.1f} (paper {weighted_average(paper_row):5.1f})\n')
    return results


bicubic_results = evaluate(['--arch', 'bicubic'], PAPER_BICUBIC)
print('paper PSNR easy/medium/hard:', PAPER_BICUBIC_PSNR)

## 2. Community TSRN checkpoint

A trained TSRN shared by a TextZoom user (yustiks) on Google Drive; reported ASTER 73.3 / 56.2 / 39.2 against the paper's 75.1 / 56.3 / 40.1.

The file comes from an unknown source, so it is loaded with `weights_only=True`: a checkpoint containing anything other than tensors and plain values is refused instead of executed. If it loads and scores close to the paper, use it as the pruning starting point and report its source together with the numbers measured here.

In [ ]:
CKPT = f'{STORE}/tsrn_community.pth'
![ -f {CKPT} ] || gdown 1j-g17V5kBmqS8giWNZuHe2d_GefAepCs -O {CKPT}

ckpt = torch.load(CKPT, map_location='cpu', weights_only=True)
print('top-level keys:', list(ckpt.keys()))
print('info:', ckpt.get('info'))
state = ckpt['state_dict_G']
print(len(state), 'tensors, first ones:', list(state)[:4])

# The STN (stn_head, tps) only runs during training; BatchNorm running statistics are not weights.
def is_inference_weight(name):
    return not name.startswith(('stn_head', 'tps')) and 'running_' not in name and 'num_batches_tracked' not in name

print(f"weights used at inference: {sum(v.numel() for k, v in state.items() if is_inference_weight(k)):,}")
print(f"all weights incl. STN:     {sum(v.numel() for k, v in state.items() if 'running_' not in k and 'num_batches_tracked' not in k):,}")

In [ ]:
tsrn_results = evaluate(['--arch', 'tsrn', '--STN', '--mask', '--resume', CKPT], PAPER_TSRN)

## 2b. Clean TSRN baseline (own training, resumable)

The community checkpoint was trained on the whole training set, so it has already seen our 868 validation images, and the original code keeps the checkpoint that is best on the test sets. This trains TSRN with the original recipe (STN + mask, MSE + gradient prior loss, Adam lr 1e-3 / beta1 0.5, batch 512, 500 epochs) on the other 95% only, and keeps the **final** weights: nothing is selected on the test sets, which are evaluated once at the end.

- **A100** (40 GB): batch 512 fits, about 2-3 hours. **T4**: the cell adds `--micro_batch 256` (still one update per 512 images), about 8-9 hours.
- Validation (PSNR + ASTER/MORAN/CRNN on the 868 images) every 5 epochs, recorded in `history.csv`.
- Saved in `experiments/baseline/<run name>/`; the baseline used by section 6 is `checkpoints/final.pth`.

**If Colab disconnects:** reconnect, run the setup cells (install → config), then this cell again. It continues from the last saved state (at most 5 epochs are redone), also on another GPU type.

In [ ]:
EXPERIMENTS = f'{STORE}/experiments'
# Batch 512 does not fit on a 16 GB T4: smaller GPUs split each batch in pieces of 256 (same update).
big_gpu = torch.cuda.get_device_properties(0).total_memory > 30e9
TRAIN_ARGS = f'--workers {min(8, os.cpu_count())}' + ('' if big_gpu else ' --micro_batch 256')
print(torch.cuda.get_device_name(0), '|', TRAIN_ARGS)

%cd {REPO}/src
!python train_tsrn_baseline.py --experiments_dir {EXPERIMENTS} {TRAIN_ARGS}

In [ ]:
# Training curve of the clean baseline (works while training runs too); saved in its figures/ folder.
import matplotlib.pyplot as plt
import pandas as pd

%cd {REPO}/src
CLEAN_DIR = !python train_tsrn_baseline.py --experiments_dir {EXPERIMENTS} --print_run_dir
CLEAN_DIR = CLEAN_DIR[-1]
history = pd.read_csv(f'{CLEAN_DIR}/history.csv')
val = history[history['val_psnr'].notna()]
READER_COLORS = {'aster': ('ASTER', '#2a78d6'), 'moran': ('MORAN', '#eb6834'), 'crnn': ('CRNN', '#1baf7a')}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), constrained_layout=True)
axes[0].plot(history['epoch'], history['loss'], color='#0b0b0b')
axes[0].set_title('a) Training loss', loc='left')
axes[1].plot(val['epoch'], val['val_psnr'], color='#0b0b0b', marker='o', markersize=3)
axes[1].set_title('b) Validation PSNR (dB)', loc='left')
for rec, (label, color) in READER_COLORS.items():
    axes[2].plot(val['epoch'], val[f'val_{rec}'], color=color, label=label)
axes[2].plot(val['epoch'], val['val_ocr_average'], color='#52514e', linestyle='--', label='average')
axes[2].set_title('c) Validation accuracy (%)', loc='left')
axes[2].legend(frameon=False)
for ax in axes:
    ax.set_xlabel('epoch')
    ax.grid(True, color='#e4e3df')
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
for ext in ('png', 'pdf'):
    fig.savefig(f'{CLEAN_DIR}/figures/training_curve.{ext}', dpi=200)
plt.show()
print(val.tail(3).to_string(index=False))

## 3. Time TSRN training (15 minutes)

Prints every 10 iterations. One epoch is 67 iterations at batch 256 (17,367 images). Estimate: seconds per iteration × 67 × 500 epochs. The paper's batch of 512 was split over 4 GPUs (128 each), so 512 does not fit on one T4. Validation (ASTER on the 3 test sets) runs every 200 iterations and is included. If you get an out-of-memory error, use `--batch_size 128`.

In [ ]:
cfg = yaml.safe_load(open(cfg_path))
cfg['TRAIN']['displayInterval'] = 10
yaml.safe_dump(cfg, open(cfg_path, 'w'), sort_keys=False)

%cd {REPO}/src
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True timeout 15m python main.py --arch tsrn --batch_size 256 --STN --mask --gradient --vis_dir tsrn_timing

## 4. Profile TSRN inference (GRU vs convolutions)

Median latency per batch, then the share of op time per category (GPU: kernel time; CPU: CPU time), at batch 1 and 64, on GPU and on CPU with 1 thread and all threads. If GRUs take most of the time, pruning convolutions alone will not make TSRN much faster. A low "GPU busy" share means the GPU mostly waits for kernel launches, so fewer channels barely help either. Takes a few minutes, mostly the CPU runs.

In [ ]:
%cd {REPO}/src
!python profile_tsrn.py --resume {CKPT}

## 5. Pruning pilot (one step, convolutions only)

1. Holds out 5% of the training set as a validation split (saved as `val_split.json`).
2. Removes 20% of the channels of every prunable convolution group (Taylor importance). GRUs, the trunk width, the input and the 4-channel output are protected.
3. Checks the output is still a 4 × 32 × 128 image.
4. Fine-tunes for 3 epochs with the unpruned model as teacher; the best epoch is picked on validation PSNR.
5. Reports params, MACs, validation CRNN accuracy and PSNR, test accuracy with all three readers, and latency, before and after.

Results are saved to Drive. Takes roughly 30 minutes, mostly the test evaluations; add `--skip_test` for a quick validation-only run.

In [ ]:
import shutil

EXPERIMENTS = f'{STORE}/experiments'
PILOT_DIR = f'{EXPERIMENTS}/pilot/tsrn_conv-only_taylor_one-step-20pct_round4_ft3ep_kd0.1_select-psnr_seed0'
# Earlier sessions saved the pilot to textzoom/pilot_ratio0.2: move it under experiments/.
if os.path.isdir(f'{STORE}/pilot_ratio0.2') and not os.path.exists(PILOT_DIR):
    os.makedirs(os.path.dirname(PILOT_DIR), exist_ok=True)
    shutil.move(f'{STORE}/pilot_ratio0.2', PILOT_DIR)

%cd {REPO}/src
!python prune_tsrn_pilot.py --resume {CKPT} --out_dir {PILOT_DIR}

## 6. RQ1 trajectory (resumable)

Seven steps of 8 channels per group (64 → 56 → … → 8, i.e. 12.5% more removed at every step), each followed by fine-tuning with the unpruned TSRN as teacher. After every step: size, MACs, latency, validation PSNR and CRNN accuracy, and test accuracy with ASTER, MORAN and CRNN. No stop rule, so the whole curve is recorded.

Settings, each combination in its own folder:
- `BASELINE`: `'clean'` = our own TSRN from section 2b (folder names start with `tsrn-clean_`); `'community'` = the community checkpoint (the first runs, folder names start with `tsrn_`).
- `RUN`: `'short'` = 3 fine-tuning epochs per step (about 1.5-3 hours on a T4); `'main'` = up to 20 epochs per step, stopping early once validation PSNR has not improved by more than 0.01 dB for 3 epochs (as in Chapter 1; about 2.5-4 hours on a T4).
- `SEED`: seed of the pruning and fine-tuning; run 0, 1 and 2 for the paper. The validation images stay the same for every seed (split seed 0, as in section 2b).

**If Colab disconnects:** reconnect, run the setup cells (install, clone, Drive, downloads/data copy, config), then run this cell again with the same settings. It continues after the last finished step; an interrupted step is redone from the previous one.

In [ ]:
BASELINE = 'clean'  # 'clean' (section 2b) or 'community'
RUN = 'main'        # 'short' or 'main'
SEED = 0            # 0, 1, 2
RUN_ARGS = {'short': '', 'main': '--epochs 20 --patience 3 --min_delta 0.01'}[RUN]

EXPERIMENTS = f'{STORE}/experiments'
%cd {REPO}/src
CLEAN_DIR = !python train_tsrn_baseline.py --experiments_dir {EXPERIMENTS} --print_run_dir
BASELINE_CKPT = {'clean': f'{CLEAN_DIR[-1]}/checkpoints/final.pth',
                 'community': f'{STORE}/tsrn_community.pth'}[BASELINE]
assert os.path.exists(BASELINE_CKPT), f'{BASELINE_CKPT} not found (finish section 2b first)'
!python prune_tsrn_trajectory.py --resume {BASELINE_CKPT} --experiments_dir {EXPERIMENTS} --seed {SEED} {RUN_ARGS}

## 7. RQ1/RQ2 figures and stop rules

a) validation PSNR change, b) test accuracy change per reader, c) MACs and latency relative to the unpruned model (1.0 = unpruned). Saved as PNG and PDF in each run's `figures/` folder; panel c uses `latency_same_session.csv` once section 8 has run.

Both stop rules are then applied to the recorded **validation** curve: each keeps the last step before its validation drop first exceeds the limit. With 868 validation images, CRNN accuracy has a standard error of about 1.6 points, so read the OCR rule with that in mind.

In [ ]:
import glob
import matplotlib.pyplot as plt
import pandas as pd

EXPERIMENTS = f'{STORE}/experiments'

READERS = {'aster': ('ASTER', '#2a78d6'), 'moran': ('MORAN', '#eb6834'), 'crnn': ('CRNN', '#1baf7a')}
LATENCY = {'gpu_b1': ('GPU, 1 image', '#6250d6', 'o'), 'gpu_b64': ('GPU, 64 images', '#008300', '^'),
           'cpu1_b1': ('CPU 1 thread, 1 image', '#e34948', 'D')}
INK, MUTED, GRID = '#0b0b0b', '#52514e', '#e4e3df'
PSNR_LIMIT, OCR_LIMIT = 0.2, 1.0  # Chapter 1 PSNR stop rule (dB) and the OCR rule of RQ2 (points)


def style(ax, title, ylabel):
    ax.set_title(title, loc='left', color=INK, fontsize=11)
    ax.set_xlabel('channels removed (%)', color=MUTED)
    ax.set_ylabel(ylabel, color=MUTED)
    ax.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    ax.tick_params(colors=MUTED)


def limit_line(ax, y, text, x_right):
    ax.axhline(y, color=MUTED, linestyle='--', linewidth=1)
    ax.annotate(text, (x_right, y), xytext=(0, 3), textcoords='offset points', ha='right', va='bottom',
                color=MUTED, fontsize=9)


def kept_step(df, column, limit):
    """Step a stop rule keeps: the last one before the validation drop first exceeds `limit`."""
    drop = df[column].iloc[0] - df[column]
    over = [i for i, d in enumerate(drop) if d > limit]
    return int(df['step'].iloc[over[0] - 1 if over else -1])


for csv_path in sorted(glob.glob(f'{EXPERIMENTS}/rq1_trajectory/*/trajectory.csv')):
    run_dir = os.path.dirname(csv_path)
    df = pd.read_csv(csv_path)
    base, x = df.iloc[0], df['removed_pct']
    tested = df[df['test_aster_average'].notna()]
    same_session = f'{run_dir}/latency_same_session.csv'
    timing = pd.read_csv(same_session) if os.path.exists(same_session) else df

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.4), constrained_layout=True)
    ax = axes[0]
    ax.plot(x, df['val_psnr'] - base['val_psnr'], color=INK, linewidth=2, marker='o', markersize=6)
    limit_line(ax, -PSNR_LIMIT, f'Chapter 1 stop rule (-{PSNR_LIMIT} dB)', x.max())
    style(ax, 'a) Validation PSNR change', 'change in PSNR (dB)')

    ax = axes[1]
    for rec, (label, color) in READERS.items():
        y = tested[f'test_{rec}_average'] - base[f'test_{rec}_average']
        ax.plot(tested['removed_pct'], y, color=color, linewidth=2, marker='o', markersize=6, label=label)
        ax.annotate(label, (tested['removed_pct'].iloc[-1], y.iloc[-1]), xytext=(6, 0),
                    textcoords='offset points', va='center', color=INK, fontsize=9)
    limit_line(ax, -OCR_LIMIT, f'-{OCR_LIMIT:g} point', tested['removed_pct'].max())
    style(ax, 'b) Test accuracy change (weighted easy/medium/hard)', 'change in accuracy (points)')
    ax.legend(frameon=False, loc='lower left')

    ax = axes[2]
    ax.plot(x, df['macs'] / base['macs'], color=MUTED, linestyle='--', linewidth=2, marker='s', markersize=6,
            label='MACs')
    for key, (label, color, marker) in LATENCY.items():
        column = f'latency_{key}_ms'
        if column in timing:
            ax.plot(timing['removed_pct'], timing[column] / timing[column].iloc[0], color=color, linewidth=2,
                    marker=marker, markersize=6, label=label)
    source = 'same session' if timing is not df else 'during the run'
    style(ax, f'c) Cost relative to unpruned (latency {source})', 'fraction of the unpruned model')
    ax.legend(frameon=False, loc='lower left')

    for ext in ('png', 'pdf'):
        fig.savefig(f'{run_dir}/figures/rq1_trajectory.{ext}', dpi=200)
    plt.show()

    print(os.path.basename(run_dir))
    for name, column, limit in (('PSNR rule (val drop <= 0.2 dB)', 'val_psnr', PSNR_LIMIT),
                                ('OCR rule  (val CRNN drop <= 1 point)', 'val_crnn', OCR_LIMIT)):
        row = df[df['step'] == kept_step(df, column, limit)].iloc[0]
        test = (' / '.join(f"{row[f'test_{r}_average']:.1f}" for r in READERS)
                if pd.notna(row['test_aster_average']) else 'not tested')
        print(f"  {name}: keeps step {int(row['step'])} ({row['removed_pct']:.1f}% channels removed, "
              f"{int(row['params']):,} params, {row['macs'] / 1e6:.0f} MMACs), test ASTER/MORAN/CRNN {test}")

## 8. Same-session latency (for the paper)

Times every model of each trajectory run back to back in this session (3 interleaved rounds, median), so all numbers come from one GPU and one CPU. Step 0 is the unpruned model each run started from (its `config.json`). Writes `latency_same_session.csv` in the run folder; re-run section 7 afterwards to use it in panel c. Run it on a **T4**, like the existing numbers, so all latencies in the paper come from the same GPU.

In [ ]:
import glob

REDO = False  # True re-times runs that already have latency_same_session.csv
EXPERIMENTS = f'{STORE}/experiments'
%cd {REPO}/src
for run_dir in sorted(glob.glob(f'{EXPERIMENTS}/rq1_trajectory/*/')):
    if REDO or not os.path.exists(f'{run_dir}/latency_same_session.csv'):
        !python measure_trajectory_latency.py --run_dir {run_dir}

## 9. Re-score every model with the three readers (RQ2, McNemar)

During the runs, validation accuracy used CRNN only. This scores every saved model of the clean runs again, image by image, with ASTER, MORAN and CRNN on the 868 validation and 4,373 test images:

- the **average of the three readers on validation** is a steadier signal for the OCR stop rule;
- **McNemar tests** on the test images say whether each pruned model reads significantly fewer words than the unpruned one.

Writes `rescore/` (one file per model, `summary.csv`) in each run folder. About 2 minutes per model (T4 or L4), roughly 45 minutes for 3 seeds. **If Colab disconnects:** run the setup cells and this cell again; models already scored are skipped.

In [ ]:
import glob

EXPERIMENTS = f'{STORE}/experiments'
RUN_DIRS = sorted(glob.glob(f'{EXPERIMENTS}/rq1_trajectory/tsrn-clean_*/'))
for run_dir in RUN_DIRS:
    print(run_dir)

%cd {REPO}/src
RUN_DIRS_ARG = ' '.join(RUN_DIRS)
!python rescore_trajectory.py --run_dirs {RUN_DIRS_ARG}

## 10. Example figure (for the paper)

Shows a few test images that the unpruned model reads right and the 87.5% model reads wrong (CRNN): input, output at 0% / 50% / 87.5% removed with what CRNN reads and the PSNR, and the target. Needs section 9 to have run. About 2 minutes, any GPU. Saved as `figures/examples_crnn.png/.pdf` in the run folder.

In [ ]:
import glob
from IPython.display import Image, display

EXPERIMENTS = f'{STORE}/experiments'
RUN_DIR = sorted(glob.glob(f'{EXPERIMENTS}/rq1_trajectory/tsrn-clean_*_seed0/'))[0]
%cd {REPO}/src
!python make_examples.py --run_dir {RUN_DIR} --steps 0 4 7 --reader crnn --per_split 2
display(Image(f'{RUN_DIR}/figures/examples_crnn.png'))